# 04 - Evaluation delle configurazioni e della pipeline originale

Usare **Esegui tutto / Run All** per ricalcolare la valutazione dai JSON salvati. Questo notebook non invia richieste a LM Studio e non rigenera obiettivi.

Confronta separatamente:

- **top_down**: gerarchie iniziali della pipeline originale e baseline congelate delle suite;
- **original_bottom_up**: output del notebook 01 in `output/top_down_bottom_up/`;
- **reference e nove alternative**: risultati del notebook 03, per ciascuna suite e ripetizione.

Per default include tutte le suite trovate. Le suite rimangono distinte nei confronti, anche quando hanno lo stesso nome di configurazione. I confronti appaiati richiedono la stessa gerarchia iniziale e descrizione. Il riferimento sperimentale viene mantenuto distinto dall'esecuzione originale, anche se i parametri coincidono.

Le run non convergenti ma completate vengono valutate e identificate; errori, run mancanti e risultati esclusi restano visibili nel report. Le medie riportano le numerosita: confronti parziali non equivalgono a risultati su tutti i 12 dataset.


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir() and (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
for directory in (PROJECT_ROOT, PROJECT_ROOT / "notebook"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

from groundtruth import GROUNDTRUTH_BY_NAME

# None = tutte le suite; altrimenti, ad esempio: ["ofat_20261021_231350"].
SUITE_IDS = None
EXPERIMENTS_ROOT = PROJECT_ROOT / "output" / "bottom_up_experiments"
SUITE_DIRS = (sorted(p.parent for p in EXPERIMENTS_ROOT.glob("*/manifest.json"))
              if SUITE_IDS is None else [EXPERIMENTS_ROOT / name for name in SUITE_IDS])
# Ogni evaluation ha una cartella nuova, senza sovrascrivere report precedenti.
REPORT_ID = datetime.now(ZoneInfo("Europe/Rome")).strftime("%Y%m%d_%H%M%S_%f")
REPORT_DIR = PROJECT_ROOT / "output" / "evaluation" / "configuration_comparison" / REPORT_ID
print("Suite incluse:", [p.name for p in SUITE_DIRS])
print("Pipeline originale:", PROJECT_ROOT / "output" / "top_down_bottom_up")
print("Report:", REPORT_DIR)


## Funzioni di valutazione incorporate

Le celle seguenti contengono la lettura e la verifica degli output, il calcolo
dei confronti e il salvataggio dei report. Sostituiscono il modulo separato
`configuration_comparison.py`. Eseguirle prima della raccolta dei risultati.
Non avviano la pipeline e non effettuano richieste ai modelli di LM Studio.

I report conservano `reused_from` e `origin_run_directory`: i risultati riutilizzati tra suite non costituiscono nuove osservazioni indipendenti.


### Controlli e lettura delle collezioni di goal


In [ ]:
from collections import Counter
import hashlib
import json
from pathlib import Path


def digest(raw):
    return hashlib.sha256(raw).hexdigest()


def baseline_key(payload):
    fields = ("description", "structuredHighLevelGoals", "structuredLowLevelGoals")
    if any(field not in payload for field in fields):
        raise ValueError("Missing verifiable initial hierarchy or description")
    return digest(json.dumps({field: payload[field] for field in fields},
                             sort_keys=True, ensure_ascii=False).encode("utf-8"))


def goal_texts(container, field):
    items = container.get(field) if isinstance(container, dict) else container
    if not isinstance(items, list):
        raise ValueError(f"Missing goal collection: {field}")
    texts = []
    for item in items:
        text = item if isinstance(item, str) else (item.get("name") or item.get("description")) if isinstance(item, dict) else None
        if not isinstance(text, str) or not text.strip():
            raise ValueError("Malformed or blank goal")
        texts.append(text.strip())
    return texts




### Raccolta degli output originali e sperimentali


In [ ]:
def collect_outputs(project_root, suite_directories):
    """Keep planned/missing runs visible; read actual JSON results, never old CSVs."""
    root = Path(project_root)
    hashes, entries, availability, baselines = {}, [], [], {}

    def read(path):
        path = Path(path).resolve()
        raw = path.read_bytes()
        checksum = digest(raw)
        if path in hashes and hashes[path] != checksum:
            raise ValueError(f"Input changed during collection: {path}")
        hashes[path] = checksum
        return json.loads(raw)

    def register_baseline(payload, path):
        key = baseline_key(payload)
        identity = (payload["name"], key)
        if identity not in baselines:
            entry = {"dataset": payload["name"], "baseline_key": key, "suite_id": "",
                     "method": "top_down", "config_id": "top_down", "repetition": 0,
                     "source_file": str(path), "payload": payload,
                     "hlg": payload["structuredHighLevelGoals"], "llg": payload["structuredLowLevelGoals"]}
            goal_texts(entry["hlg"], "goals")
            goal_texts(entry["llg"], "low_level_goals")
            entries.append(entry)
            baselines[identity] = entry
            availability.append({k: v for k, v in entry.items() if k not in {"payload", "hlg", "llg"}} | {"status": "completed"})
        return key

    for path in sorted((root / "output" / "top_down").glob("*_top_down.json")):
        register_baseline(read(path), path)
    for directory in map(Path, suite_directories):
        manifest = read(directory / "manifest.json")
        for baseline in manifest["baselines"]:
            source = directory / baseline["snapshot"]
            payload = read(source)
            if hashes[source.resolve()] != baseline["sha256"]:
                raise ValueError(f"Frozen baseline altered: {source}")
            key = register_baseline(payload, source)
            for config in manifest["configurations"]:
                for repetition in range(1, manifest["repetitions"] + 1):
                    identity = {"dataset": baseline["dataset"], "baseline_key": key,
                        "suite_id": manifest["suite_id"], "config_id": config["config_id"],
                        "method": f"{manifest['suite_id']}/{config['config_id']}", "repetition": repetition}
                    job = directory / "runs" / config["config_id"] / f"repeat_{repetition:03d}" / baseline["dataset_id"]
                    attempts = sorted(job.glob("attempt_*/run.json"))
                    if not attempts:
                        availability.append({**identity, "status": "not_started"})
                        continue
                    run_path = attempts[-1]
                    record = read(run_path)
                    row = {**identity, "status": record["status"], "error": record.get("error"),
                           "source_file": str(run_path), "elapsed_seconds": record.get("elapsed_seconds"),
                           "reused_from": record.get("reused_from"),
                           "origin_run_directory": record.get("origin_run_directory")}
                    if record["status"] != "completed":
                        availability.append(row)
                        continue
                    result_path = run_path.parent / "result.json"
                    result = read(result_path)
                    if hashes[result_path.resolve()] != record["result_sha256"]:
                        raise ValueError(f"Result hash mismatch: {result_path}")
                    if (result.get("baselineSha256") != baseline["sha256"] or result.get("name") != baseline["dataset"]
                            or result.get("experiment") != config or result.get("repetition") != repetition):
                        raise ValueError(f"Result provenance mismatch: {result_path}")
                    goal_texts(result["finalHighLevelGoals"], "goals")
                    goal_texts(result["finalLowLevelGoals"], "low_level_goals")
                    entries.append({**identity, "source_file": str(result_path), "payload": result,
                                    "reused_from": record.get("reused_from"),
                                    "origin_run_directory": record.get("origin_run_directory"),
                                    "hlg": result["finalHighLevelGoals"], "llg": result["finalLowLevelGoals"]})
                    availability.append({**row, "source_file": str(result_path)})
    for path in sorted((root / "output" / "top_down_bottom_up").glob("*.json")):
        data = read(path)
        identity = {"dataset": data.get("name"), "method": "original_bottom_up",
                    "config_id": "original_bottom_up", "suite_id": "", "repetition": 0, "source_file": str(path)}
        try:
            if data.get("bottomUpError"):
                raise ValueError(data["bottomUpError"])
            key = baseline_key(data)
            if (data["name"], key) not in baselines:
                raise ValueError("Original output has a different/unavailable baseline")
            if any(e["method"] == "original_bottom_up" and e["dataset"] == data["name"] and e["baseline_key"] == key for e in entries):
                raise ValueError("Duplicate original output for the same baseline")
            goal_texts(data["finalHighLevelGoals"], "goals")
            goal_texts(data["finalLowLevelGoals"], "low_level_goals")
            entries.append({**identity, "baseline_key": key, "payload": data,
                            "hlg": data["finalHighLevelGoals"], "llg": data["finalLowLevelGoals"]})
            availability.append({**identity, "baseline_key": key, "status": "completed"})
        except (KeyError, ValueError, TypeError) as exc:
            availability.append({**identity, "status": "excluded", "error": str(exc)})
    for (dataset, key) in baselines:
        if not any(e["method"] == "original_bottom_up" and e["dataset"] == dataset
                   and e["baseline_key"] == key for e in entries):
            availability.append({"dataset": dataset, "baseline_key": key,
                "method": "original_bottom_up", "config_id": "original_bottom_up",
                "suite_id": "", "repetition": 0, "status": "unavailable",
                "error": "No eligible original bottom-up output for this baseline"})
    return {"entries": entries, "availability": availability, "input_hashes": hashes}




### Metriche e confronti appaiati


In [ ]:
def evaluate_outputs(collection, references, score_goals):
    import pandas as pd
    rows, iteration_rows, matches = [], [], []
    for entry in collection["entries"]:
        identity = {k: entry[k] for k in ("dataset", "baseline_key", "suite_id", "method", "config_id", "repetition", "source_file")}
        if identity["dataset"] not in references:
            raise ValueError(f"No ground truth: {identity['dataset']}")
        cycle = entry["payload"].get("globalGoalCycle", {})
        counts = Counter()
        for trace in cycle.get("iterations", []):
            for evaluation in trace.get("evaluations", {}).values():
                votes = evaluation.get("critic_votes", {}).get("documentary", [])
                valid = [v for v in votes if v["status"] == "VALID"]
                winner = max(Counter(v["decision"] for v in valid).values(), default=0)
                counts.update(branches=1, votes=len(votes), valid_votes=len(valid), quorum=int(winner >= 2))
        status = {"reused_from": entry.get("reused_from"),
                  "origin_run_directory": entry.get("origin_run_directory"),
                  "converged": cycle.get("converged"), "completed_iterations": cycle.get("completed_iterations"),
                  "stop_reason": cycle.get("stop_reason"), "warning_count": len(cycle.get("warnings", [])),
                  "quorum_rate": counts["quorum"] / counts["branches"] if counts["branches"] else None,
                  "valid_vote_rate": counts["valid_votes"] / counts["votes"] if counts["votes"] else None}

        def score(hlg, llg, iteration=None):
            for level, container, field, ref_field in (("HLG", hlg, "goals", "highLevelGoals"),
                                                      ("LLG", llg, "low_level_goals", "lowLevelGoals")):
                preds = goal_texts(container, field)
                refs = references[identity["dataset"]][ref_field]
                result = dict(score_goals(preds, refs))
                assigned = result.pop("matches", [])
                row = {**identity, **status, "level": level, "generated_goal_count": len(preds),
                       "reference_goal_count": len(refs), **result}
                if iteration is None:
                    rows.append(row)
                    matches.extend({**identity, "level": level, **match} for match in assigned)
                else:
                    iteration_rows.append({**row, "iteration": iteration})
        score(entry["hlg"], entry["llg"])
        for trace in cycle.get("iterations", []):
            if trace.get("checkpoint_complete") is not False:
                score(trace["high_level_goals"], trace["low_level_goals"], trace["iteration"])
    metrics = pd.DataFrame(rows)
    values = ["precision", "recall", "f1"]
    # Each dataset has the same weight even when repetition counts differ.
    dataset_means = metrics.groupby(["method", "config_id", "suite_id", "dataset", "baseline_key", "level"], dropna=False)[values].mean().reset_index()
    macro = dataset_means.groupby(["method", "level"])[values].agg(["mean", "std", "count"])
    macro.columns = ["_".join(c) for c in macro.columns]
    deltas = []
    for comparator in ("top_down", "original_bottom_up", "reference"):
        if comparator == "reference":
            candidates = metrics[metrics.suite_id != ""]
            target = candidates[candidates.config_id == "reference"]
            candidates = candidates[candidates.config_id != "reference"]
            keys = ["dataset", "baseline_key", "level", "suite_id", "repetition"]
        else:
            target = metrics[metrics.method == comparator]
            candidates = metrics[metrics.method != comparator]
            keys = ["dataset", "baseline_key", "level"]
        paired = candidates.merge(target[keys + values], on=keys, how="left", validate="many_to_one", suffixes=("", "_comparator"))
        paired["comparator"] = comparator
        paired["pair_available"] = paired["f1_comparator"].notna()
        for metric in values:
            paired[f"delta_{metric}"] = paired[metric] - paired[f"{metric}_comparator"]
        deltas.append(paired)
    deltas = pd.concat(deltas, ignore_index=True)
    paired_dataset = deltas[deltas.pair_available].groupby(["method", "comparator", "dataset", "baseline_key", "level"])[
        values + [f"{v}_comparator" for v in values] + [f"delta_{v}" for v in values]].mean().reset_index()
    paired_macro = paired_dataset.groupby(["method", "comparator", "level"])[
        values + [f"{v}_comparator" for v in values] + [f"delta_{v}" for v in values]].agg(["mean", "std", "count"])
    paired_macro.columns = ["_".join(c) for c in paired_macro.columns]
    availability = pd.DataFrame(collection["availability"])
    coverage = availability.groupby(["method", "status"]).size().rename("runs").reset_index()
    variability = metrics.groupby(["method", "dataset", "baseline_key", "level"])[values].agg(["mean", "std", "count"])
    variability.columns = ["_".join(c) for c in variability.columns]
    return {"availability": availability, "coverage": coverage, "metrics": metrics,
            "dataset_means": dataset_means, "macro_metrics": macro.reset_index(),
            "paired_deltas": deltas, "paired_macro": paired_macro.reset_index(),
            "repeat_variability": variability.reset_index(), "iteration_metrics": pd.DataFrame(iteration_rows),
            "matches": pd.DataFrame(matches)}




### Salvataggio dei report e della provenienza


In [ ]:
def export_comparison(collection, tables, references, destination, metric_configuration):
    """Refuse mixed versions when an input is overwritten during evaluation."""
    changed = [str(path) for path, checksum in collection["input_hashes"].items()
               if not path.exists() or digest(path.read_bytes()) != checksum]
    if changed:
        raise RuntimeError(f"Inputs changed during evaluation; rerun collection: {changed}")
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=False)
    for name, table in tables.items():
        table.to_csv(destination / f"{name}.csv", index=False, encoding="utf-8-sig")
    provenance = {"metric": metric_configuration,
        "sources": {str(p): checksum for p, checksum in collection["input_hashes"].items()},
        "ground_truth_sha256": digest(json.dumps(references, sort_keys=True, ensure_ascii=False).encode()),
        "ground_truth_snapshot": references,
        "pairing": "Identical dataset, initial hierarchy and description; reference also matches suite and repetition.",
        "aggregation": "Mean repetitions within dataset, then macro across datasets; paired tables use identical cohorts.",
        "status_policy": "All completed final outputs including nonconvergent; missing/failed/excluded only in availability.",
        "original_configuration": {e["dataset"]: e["payload"].get("bottomUpModelConfiguration", {})
                                   for e in collection["entries"] if e["method"] == "original_bottom_up"}}
    (destination / "evaluation_manifest.json").write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8")
    return destination


## 1. Lettura e verifica dei risultati

Legge i JSON effettivi e verifica gli hash degli esperimenti. Per ogni run usa l'ultimo tentativo, senza mescolare checkpoint di tentativi precedenti. Le baseline top-down uguali sono deduplicate; se differiscono vengono mantenute come coorti separate. Le righe della pipeline originale sono ammesse solo quando gli input incorporati corrispondono a una baseline disponibile.

Se le run sono ancora in corso, il report fotografa i risultati disponibili al momento della lettura. Se un file letto viene modificato durante la valutazione, l'export si ferma: rieseguire la raccolta e il calcolo per avere input coerenti.


In [ ]:
collection = collect_outputs(PROJECT_ROOT, SUITE_DIRS)
availability_df = pd.DataFrame(collection["availability"])
display(availability_df)
display(availability_df.groupby(["method", "status"]).size().rename("run_count").reset_index())
print("Gerarchie finali valutabili:", len(collection["entries"]))
print("Dataset con ground truth:", len(GROUNDTRUTH_BY_NAME))


## 2. Metrica comune a tutti gli output

Riutilizza la metrica del notebook 02: `bert-base-uncased`, attention-mask mean pooling, testo originale del nome del goal (descrizione come fallback), matching Hungarian uno-a-uno. Mantiene solo gli abbinamenti con cosine >= **0.65**. Precision e Recall sono soft: somma delle cosine accettate divisa per numero di goal generati e di riferimento; F1 e la media armonica. Nessun preprocessing linguistico.

La soglia di evaluation rimane fissa per tutte le configurazioni, indipendentemente dalle soglie operative dell'esperimento E2. Modello su CPU per consentire la valutazione anche mentre LM Studio occupa la GPU. Al primo utilizzo Transformers potrebbe scaricare il modello.


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

BERT_MODEL_NAME = "bert-base-uncased"
SIMILARITY_THRESHOLD = 0.65
BERT_BATCH_SIZE = 16
device = torch.device("cpu")
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
model = AutoModel.from_pretrained(BERT_MODEL_NAME).to(device)
model.eval()

class PaperAlignedGoalEvaluator:
    def __init__(self, tokenizer, model, device, batch_size=16, similarity_threshold=0.65):
        self.tokenizer = tokenizer
        self.model = model
        self.device = device
        self.batch_size = batch_size
        self.embedding_cache = {}
        self.similarity_threshold = similarity_threshold

    def _encode_missing(self, raw_texts):
        missing = list(dict.fromkeys(text for text in raw_texts if text not in self.embedding_cache))
        if not missing:
            return

        total_batches = (len(missing) + self.batch_size - 1) // self.batch_size
        for start in tqdm(
            range(0, len(missing), self.batch_size),
            total=total_batches,
            desc="BERT batches",
        ):
            raw_batch = missing[start:start + self.batch_size]

            encoded_input = self.tokenizer(
                raw_batch,
                padding=True,
                truncation=True,
                return_tensors="pt",
            ).to(self.device)

            with torch.inference_mode():
                model_output = self.model(**encoded_input)

            token_embeddings = model_output.last_hidden_state
            attention_mask = encoded_input["attention_mask"]
            input_mask_expanded = (
                attention_mask.unsqueeze(-1)
                .expand(token_embeddings.size())
                .float()
            )

            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
            sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
            pooled = (sum_embeddings / sum_mask).detach().cpu()

            for text, embedding in zip(raw_batch, pooled):
                self.embedding_cache[text] = embedding

    def encode_texts(self, texts):
        self._encode_missing(texts)
        return torch.stack([self.embedding_cache[text] for text in texts])

    def compute_similarity(self, generated_goals, reference_goals):
        gen_embeddings = self.encode_texts(generated_goals)
        ref_embeddings = self.encode_texts(reference_goals)
        gen_norm = torch.nn.functional.normalize(gen_embeddings, p=2, dim=1)
        ref_norm = torch.nn.functional.normalize(ref_embeddings, p=2, dim=1)
        return torch.mm(gen_norm, ref_norm.transpose(0, 1)).numpy()

    def calculate_soft_f1(self, preds, refs):
        if not preds or not refs:
            return {"precision": 0.0, "recall": 0.0, "f1": 0.0, "matched_goal_count": 0}

        sim_matrix = self.compute_similarity(preds, refs)
        row_ind, col_ind = linear_sum_assignment(-sim_matrix)
        matched_similarities = sim_matrix[row_ind, col_ind]
        accepted = matched_similarities[matched_similarities >= self.similarity_threshold]
        total_similarity = float(accepted.sum())

        precision = total_similarity / len(preds)
        recall = total_similarity / len(refs)
        f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
        return {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "matched_goal_count": int(len(accepted)),
        }


evaluator = PaperAlignedGoalEvaluator(tokenizer, model, device,
    batch_size=BERT_BATCH_SIZE, similarity_threshold=SIMILARITY_THRESHOLD)

def score_with_matches(preds, refs):
    score = evaluator.calculate_soft_f1(preds, refs)
    matches = []
    if preds and refs:
        matrix = evaluator.compute_similarity(preds, refs)
        row_indices, col_indices = linear_sum_assignment(-matrix)
        for i, j in zip(row_indices, col_indices):
            cosine = float(matrix[i, j])
            matches.append({"generated_index": int(i), "reference_index": int(j),
                            "generated_text": preds[i], "reference_text": refs[j],
                            "cosine": cosine, "accepted": cosine >= SIMILARITY_THRESHOLD})
    return {**score, "matches": matches}

METRIC_CONFIGURATION = {
    "model": BERT_MODEL_NAME, "resolved_revision": getattr(model.config, "_commit_hash", None),
    "pooling": "attention-mask mean pooling", "matching": "Hungarian one-to-one",
    "threshold": SIMILARITY_THRESHOLD, "goal_text": "name, description fallback",
    "preprocessing": "none", "scoring": "soft precision/recall; harmonic F1",
}
tables = evaluate_outputs(collection, GROUNDTRUTH_BY_NAME, score_with_matches)


## 3. Metriche per dataset e riepiloghi

`metrics` contiene tutte le esecuzioni finali per HLG/LLG, con convergenza, warning, quorum e validita dei voti quando disponibili. `dataset_means` media le ripetizioni prima della macro-media. `repeat_variability` separa la variabilita tra ripetizioni dello stesso dataset dalla deviazione standard tra dataset riportata nei riepiloghi macro. Con una sola ripetizione la prima non e stimabile.


In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(tables["metrics"])
    display(tables["macro_metrics"])
    display(tables["repeat_variability"])

# Una colonna per ciascun metodo, una riga per dataset/livello/coorte iniziale.
f1_by_dataset = tables["dataset_means"].pivot_table(
    index=["dataset", "baseline_key", "level"], columns="method", values="f1")
display(f1_by_dataset)
tables["f1_by_dataset"] = f1_by_dataset.reset_index()


## 4. Confronti appaiati: originale, top-down e riferimento

Ogni output viene confrontato con **top_down** e **original_bottom_up** sullo stesso dataset, baseline e livello. Ogni alternativa sperimentale viene anche confrontata con **reference** della stessa suite e ripetizione.

`pair_available=False` indica un confronto mancante: non viene trasformato in un delta pari a zero. `paired_macro` riporta medie e delta su coppie effettivamente disponibili e sulla stessa coorte. Una tabella macro descrittiva su dataset differenti non basta a stabilire quale metodo sia migliore.


In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(tables["paired_macro"])
    display(tables["paired_deltas"][["dataset", "method", "repetition", "level", "comparator",
                                     "pair_available", "f1", "f1_comparator", "delta_f1"]])
missing_pairs = tables["paired_deltas"][~tables["paired_deltas"]["pair_available"]]
print("Confronti senza coppia disponibile:", len(missing_pairs))


## 5. Export di tutti i confronti

Salva i CSV e un manifest con hash degli input, metrica, ground truth e configurazione della pipeline originale. Le gerarchie originali e gli output delle suite non vengono modificati. Non occorre eseguire le celle di evaluation del notebook 03: i risultati sono ricalcolati direttamente dai JSON.


In [ ]:
export_comparison(collection, tables, GROUNDTRUTH_BY_NAME, REPORT_DIR, METRIC_CONFIGURATION)
print("Report salvato:", REPORT_DIR)
for path in sorted(REPORT_DIR.iterdir()):
    print("-", path.name)


## 6. Grafici esportabili

Le heatmap mostrano la F1 per dataset e metodo, separatamente per HLG e LLG. I valori mancanti rimangono vuoti. Le macro-medie nel secondo grafico riportano le numerosita nei CSV. Le traiettorie delle iterazioni completate sono esportate in `iteration_metrics.csv`; per la baseline iniziale si usano le righe `top_down` di `metrics.csv`.


In [ ]:
import numpy as np

for level in ("HLG", "LLG"):
    frame = tables["dataset_means"]
    frame = frame[frame.level == level].copy()
    # L'hash breve distingue dataset con input iniziali diversi.
    frame["dataset_cohort"] = frame["dataset"] + " [" + frame["baseline_key"].str[:8] + "]"
    pivot = frame.pivot(index="dataset_cohort", columns="method", values="f1")
    fig, ax = plt.subplots(figsize=(max(12, .8 * len(pivot.columns)), max(5, .45 * len(pivot))))
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad("#eeeeee")
    image = ax.imshow(np.ma.masked_invalid(pivot.to_numpy(dtype=float)), vmin=0, vmax=1, aspect="auto", cmap=cmap)
    ax.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=65, ha="right", fontsize=8)
    ax.set_yticks(range(len(pivot.index)), pivot.index, fontsize=8)
    ax.set_title(f"{level}: F1 soft per dataset e configurazione")
    fig.colorbar(image, ax=ax, label="F1 soft")
    fig.tight_layout()
    fig.savefig(REPORT_DIR / f"f1_heatmap_{level}.png", dpi=180, bbox_inches="tight")
    plt.show()

fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
for ax, level in zip(axes, ("HLG", "LLG")):
    frame = tables["macro_metrics"]
    frame = frame[frame.level == level]
    ax.bar(range(len(frame)), frame.f1_mean)
    ax.set_xticks(range(len(frame)), frame.method, rotation=70, ha="right", fontsize=8)
    ax.set_title(f"{level} - macro F1 (verificare f1_count nel CSV)")
    ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(REPORT_DIR / "macro_f1_comparison.png", dpi=180, bbox_inches="tight")
plt.show()
print("Evaluation completata:", REPORT_DIR)


Output principali:

- `availability.csv`, `coverage.csv`: run incluse, mancanti, fallite o escluse.
- `metrics.csv`, `dataset_means.csv`, `f1_by_dataset.csv`: tutte le metriche HLG/LLG e confronto per progetto.
- `macro_metrics.csv`: medie descrittive con numerosita.
- `paired_deltas.csv`, `paired_macro.csv`: confronto con top-down, bottom-up originale e riferimento sperimentale.
- `repeat_variability.csv`: stabilita tra ripetizioni dello stesso dataset.
- `iteration_metrics.csv`, `matches.csv`: dettaglio delle iterazioni e degli abbinamenti finali.
- `evaluation_manifest.json`: provenienza degli input e delle ground truth, metrica e policy della pipeline originale.
- `f1_heatmap_HLG.png`, `f1_heatmap_LLG.png`, `macro_f1_comparison.png`: grafici esportabili.

Per aggiornare il confronto dopo nuove run, eseguire nuovamente tutto il notebook: viene creata una nuova cartella di report.
